# 03 - Modeling

Baseline: the trivial "always predict no readmission" model, then logistic regression. Improve: Random Forest and Gradient Boosting, both compensating for class imbalance (`class_weight="balanced"` for RF/LogReg, balanced `sample_weight` for GB, since `GradientBoostingClassifier` has no `class_weight`). Compare ROC-AUC and recall, then save the best model.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.config import PROCESSED_DATA_DIR, TARGET_COLUMN
from src.models.train import split_data, build_pipeline, build_gb_pipeline, train_and_save
from src.models.baseline import evaluate_always_no, build_logistic_pipeline
from src.models.evaluate import evaluate_model

df = pd.read_csv(PROCESSED_DATA_DIR / "diabetic_data_clean.csv")
X_train, X_test, y_train, y_test = split_data(df)
X_train.shape, X_test.shape

((56351, 44), (14088, 44))

## Baseline 1: always predict "no"

In [2]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score

dummy_result = evaluate_always_no(y_train, y_test)
dummy_metrics = {
    "roc_auc": roc_auc_score(y_test, dummy_result["y_proba"]),
    "recall": recall_score(y_test, dummy_result["y_pred"]),
    "precision": precision_score(y_test, dummy_result["y_pred"], zero_division=0),
}
dummy_metrics

{'roc_auc': 0.5, 'recall': 0.0, 'precision': 0.0}

ROC-AUC of 0.50 and recall of 0 — it never flags a single readmission. This is the floor every real model has to beat.

## Baseline 2: Logistic Regression

In [3]:
logreg_pipeline = build_logistic_pipeline(X_train)
logreg_pipeline.fit(X_train, y_train)
logreg_metrics = evaluate_model(logreg_pipeline, X_test, y_test)
{"roc_auc": logreg_metrics["roc_auc"], "recall": logreg_metrics["classification_report"]["1"]["recall"]}

{'roc_auc': 0.6372583684787327, 'recall': 0.528196981731533}

## Improve: Random Forest

In [4]:
rf_pipeline = build_pipeline(X_train)
rf_pipeline = train_and_save(rf_pipeline, X_train, y_train, "random_forest.joblib")
rf_metrics = evaluate_model(rf_pipeline, X_test, y_test)
{"roc_auc": rf_metrics["roc_auc"], "recall": rf_metrics["classification_report"]["1"]["recall"]}

{'roc_auc': 0.6388269948614113, 'recall': 0.5520254169976172}

## Improve: Gradient Boosting

In [5]:
gb_pipeline = build_gb_pipeline(X_train)
gb_pipeline = train_and_save(gb_pipeline, X_train, y_train, "gradient_boosting.joblib")
gb_metrics = evaluate_model(gb_pipeline, X_test, y_test)
{"roc_auc": gb_metrics["roc_auc"], "recall": gb_metrics["classification_report"]["1"]["recall"]}

{'roc_auc': 0.6399810831186863, 'recall': 0.5472597299444003}

## Comparison

In [6]:
comparison = pd.DataFrame([
    {"model": "Always no (baseline)", **dummy_metrics},
    {"model": "Logistic Regression", "roc_auc": logreg_metrics["roc_auc"],
     "recall": logreg_metrics["classification_report"]["1"]["recall"],
     "precision": logreg_metrics["classification_report"]["1"]["precision"]},
    {"model": "Random Forest", "roc_auc": rf_metrics["roc_auc"],
     "recall": rf_metrics["classification_report"]["1"]["recall"],
     "precision": rf_metrics["classification_report"]["1"]["precision"]},
    {"model": "Gradient Boosting", "roc_auc": gb_metrics["roc_auc"],
     "recall": gb_metrics["classification_report"]["1"]["recall"],
     "precision": gb_metrics["classification_report"]["1"]["precision"]},
]).set_index("model").round(3)
comparison

,roc_auc,recall,precision
model,,,
Always no (baseline),0.500,0.000,0.000
Logistic Regression,0.637,0.528,0.139
Random Forest,0.639,0.552,0.137
Gradient Boosting,0.640,0.547,0.137


Whichever of Random Forest / Gradient Boosting has the higher ROC-AUC is saved as `models/best_model.joblib` for `04_evaluation` and the Streamlit app to load.

In [7]:
import shutil
from src.config import MODELS_DIR

best_name = "random_forest.joblib" if rf_metrics["roc_auc"] >= gb_metrics["roc_auc"] else "gradient_boosting.joblib"
shutil.copy(MODELS_DIR / best_name, MODELS_DIR / "best_model.joblib")
print(f"best model: {best_name}")

best model: gradient_boosting.joblib
